# Active regions (AR): standard full pipeline with demo final prediction

This notebook follows the same logical pipeline as the original thesis notebooks:

1. project setup;
2. dataset configuration;
3. training/test data discovery;
4. temporal frame validation;
5. preprocessing and generators;
6. model construction;
7. training stage;
8. training curves and test evaluation;
9. final prediction and post-processing.

The full training dataset is **not included** in this GitHub repository.  
Therefore, the notebook keeps the training pipeline structure, but the heavy training/evaluation stages are disabled by default.  
For the final prediction stage, it loads the saved trained models from `trained_models/` and the compact demo prediction dataset from `data/`.

## Repository structure

Expected repository structure:

```text
repo/
├── src/
│   ├── model_scss_net.py
│   ├── v2_modified_model_scss_net.py
│   └── metrics.py
│
├── trained_models/
│   ├── AR_base_model.keras
│   └── AR_ConvLstm_model.keras
│
├── data/
│   └── AR_predict_2021/
│       ├── images_AR.zip
│       ├── masks_AR.zip
│       └── sequences_exp_AR.zip
│
└── notebooks/
    └── this notebook
```

The original full training folders are intentionally not stored in the repository.

## 1. Setup

This replaces the original Colab/Google Drive setup.  
The notebook uses only relative paths inside the repository.

In [ ]:
from pathlib import Path
import sys
import os
import re
import json
import random
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

SRC_DIR = PROJECT_ROOT / "src"
TRAINED_MODELS_DIR = PROJECT_ROOT / "trained_models"
DEMO_DATA_DIR = PROJECT_ROOT / "data" / "AR_predict_2021"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUT_DIR = OUTPUT_ROOT / "AR_standard_pipeline_demo"

sys.path.insert(0, str(SRC_DIR))

OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("TRAINED_MODELS_DIR:", TRAINED_MODELS_DIR)
print("DEMO_DATA_DIR:", DEMO_DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

**Explanation:**  
In the original notebook this block mounted Google Drive and sometimes copied data to `/content`.  
Here the same role is handled by local repository-relative paths.

## 2. Dataset-specific configuration

This cell defines the standard configuration for the current structure type and keeps the same variables as the original training notebooks.

In [ ]:
# =========================
# DATASET-SPECIFIC CONFIG
# =========================
PHENOMENON = "AR"
CHANNEL = "171"
SOURCE_FOLDER = "171_temporal_T3"
TRAIN_DIR_NAME = "171_train"
TEST_DIR_NAME = "171_test"
FRAMES_DIR_NAME = "ar_frames"
EXP_DIR_NAME = "AR_exp"
PREFIX = "AR_171"

# =========================
# FULL DATASET PATHS
# =========================
# The full dataset is not included in the repository.
# If you want to run full training, place the original dataset here or change DATA_ROOT.
DATA_ROOT = PROJECT_ROOT / "full_experiment_data_not_included"
EXPERIMENT_ROOT = PROJECT_ROOT / "full_experiment_not_included"

TRAIN_DIR = DATA_ROOT / SOURCE_FOLDER / TRAIN_DIR_NAME
TEST_DIR = DATA_ROOT / SOURCE_FOLDER / TEST_DIR_NAME
FRAMES_DIR = DATA_ROOT / SOURCE_FOLDER / FRAMES_DIR_NAME

FINAL_ROOT = DATA_ROOT / EXP_DIR_NAME
FINAL_IMAGES_DIR = FINAL_ROOT / "images"
FINAL_MASKS_DIR = FINAL_ROOT / "masks"
FINAL_SEQUENCES_DIR = FINAL_ROOT / "sequences"

# =========================
# DEMO FINAL PREDICTION DATA
# =========================
DEMO_IMAGES_ZIP = DEMO_DATA_DIR / "images_AR.zip"
DEMO_MASKS_ZIP = DEMO_DATA_DIR / "masks_AR.zip"
DEMO_SEQUENCES_ZIP = DEMO_DATA_DIR / "sequences_exp_AR.zip"

# =========================
# MODEL / TRAINING CONFIG
# =========================
IMG_SIZE = 256
CHANNELS = 1

HIST_T = 3
USE_CURRENT_FRAME = True
T_STEPS = HIST_T + int(USE_CURRENT_FRAME)

FILTERS = 32
LAYERS = 4
BATCH_NORM = True
DROP_PROB = 0.3

CONVLSTM_FILTERS = 32
CONVLSTM_KERNEL = (3, 3)
CONVLSTM_DROPOUT = 0.0
CONVLSTM_REC_DROPOUT = 0.0

BATCH_SIZE_BASELINE = 8
BATCH_SIZE_CONVLSTM = 4
EPOCHS = 150
LR = 1e-4

THRESHOLD = 0.25

AUGMENT_TRAIN = True
AUG_HFLIP_PROB = 0.5
AUG_VFLIP_PROB = 0.5
AUG_EXTRA_ROT90_PROB = 0.5
AUG_GAMMA_RANGE = (0.8, 1.2)
AUG_BRIGHTNESS_RANGE = (0.9, 1.1)

RUN_FULL_TRAINING = False
RUN_FULL_TEST_EVALUATION = False

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print("PHENOMENON:", PHENOMENON)
print("CHANNEL:", CHANNEL)
print("T_STEPS:", T_STEPS)
print("Standard filters:", FILTERS)
print("Standard layers:", LAYERS)
print("ConvLSTM filters:", CONVLSTM_FILTERS)
print("Final threshold:", THRESHOLD)

**Explanation:**  
These values define the standard configuration.  
Other experiments from the thesis were created by changing only selected values, such as dataset paths, number of filters, number of layers or final prediction dataset.

## 3. Import project code

The model definitions and metrics are loaded from the `src/` folder.

In [ ]:
from model_scss_net import scss_net
from v2_modified_model_scss_net import scss_net_convlstm_early
from metrics import dice_soft, iou_soft, dice_np, iou_np, bce_dice_loss

LOSS_FN = bce_dice_loss

print("Using loss:", LOSS_FN.__name__)

**Explanation:**  
Only the actually used loss and metrics are imported.  
Experimental losses that were not used in the final standard pipeline are not required.

## 4. Training and test data discovery

This section corresponds to the original `DATA DISCOVERY DIRECTLY FROM FOLDERS` block.

Since the full dataset is not included, this block is written so that it does not fail when the full data folders are missing.  
If the full data are placed into `DATA_ROOT`, the same logic can be used to build the training and test tables.

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def list_image_files(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])


def rotation_info(stem: str):
    for suffix, k in [("_r90", 1), ("_r180", 2), ("_r270", 3)]:
        if stem.lower().endswith(suffix):
            return stem[:-len(suffix)], k
    return stem, 0


def find_mask_for_image(img_path: Path, mask_dir: Path):
    candidates = [
        mask_dir / f"{img_path.stem}.png",
        mask_dir / f"{img_path.stem}_mask.png",
        mask_dir / f"{img_path.stem}_label.png",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def build_pair_df_from_source(img_dir: Path, mask_dir: Path, source_name: str):
    rows = []
    for img_path in list_image_files(img_dir):
        base_stem, rot_k = rotation_info(img_path.stem)
        mask_path = find_mask_for_image(img_path, mask_dir)
        if mask_path is None:
            continue
        rows.append({
            "stem": img_path.stem,
            "base_stem": base_stem,
            "rotation_k": rot_k,
            "source": source_name,
            "image_path": img_path,
            "mask_path": mask_path,
        })
    return rows


def build_train_df(train_dir: Path):
    """
    AR train structure may include sources such as SPoCA and Zooniverse.
    This generic discovery function searches recursively for folders named 'imgs'
    and pairs them with sibling folders whose name starts with 'mask'.
    """
    rows = []
    if not train_dir.exists():
        print("[INFO] Full train directory not found:", train_dir)
        return pd.DataFrame(rows)

    img_dirs = sorted([p for p in train_dir.rglob("imgs") if p.is_dir()])
    for img_dir in img_dirs:
        parent = img_dir.parent
        possible_mask_dirs = [p for p in parent.iterdir() if p.is_dir() and p.name.lower().startswith("mask")]
        for mask_dir in possible_mask_dirs:
            rows.extend(build_pair_df_from_source(img_dir, mask_dir, parent.name))

    return pd.DataFrame(rows)


def build_test_df(test_dir: Path):
    rows = []
    if not test_dir.exists():
        print("[INFO] Full test directory not found:", test_dir)
        return pd.DataFrame(rows)

    img_candidates = [
        test_dir / "imgs",
        test_dir / "images",
        test_dir,
    ]
    mask_candidates = [
        test_dir / "masks",
        test_dir / "mask",
        test_dir / "masks(spoca)",
        test_dir / "masks(region_growth)",
    ]

    img_dir = next((p for p in img_candidates if p.exists()), None)
    mask_dir = next((p for p in mask_candidates if p.exists()), None)

    if img_dir is None or mask_dir is None:
        print("[WARN] Could not detect test img/mask directories")
        return pd.DataFrame(rows)

    rows.extend(build_pair_df_from_source(img_dir, mask_dir, "test"))
    return pd.DataFrame(rows)


train_df = build_train_df(TRAIN_DIR)
test_df = build_test_df(TEST_DIR)

print("Train pairs:", len(train_df))
print("Test pairs:", len(test_df))

if len(train_df):
    display(train_df.head())
if len(test_df):
    display(test_df.head())

**Explanation:**  
In the real pipeline this stage creates `train_df` and `test_df`, where each row stores the path to an image, its mask, source information and rotation metadata.  
In this GitHub demo, these tables remain empty unless the full dataset is added manually.

## 5. Temporal frame validation

This block checks whether each sample has the required previous temporal frames `input_1.png`, `input_2.png`, `input_3.png`.  
The target image is then appended as the fourth frame.

In [ ]:
def frame_dir_for_base(base_stem: str):
    return FRAMES_DIR / base_stem


def has_temporal_frames(base_stem: str):
    d = frame_dir_for_base(base_stem)
    return all((d / f"input_{i}.png").exists() for i in range(1, HIST_T + 1))


if len(train_df) > 0 and "base_stem" in train_df.columns:
    train_df["has_temporal"] = train_df["base_stem"].apply(has_temporal_frames)
    print("Train temporal ok:", int(train_df["has_temporal"].sum()), "/", len(train_df))
    train_df = train_df[train_df["has_temporal"]].reset_index(drop=True)
else:
    print("[INFO] No full train data available; temporal validation is demonstrated only.")

if len(test_df) > 0 and "base_stem" in test_df.columns:
    test_df["has_temporal"] = test_df["base_stem"].apply(has_temporal_frames)
    print("Test temporal ok:", int(test_df["has_temporal"].sum()), "/", len(test_df))
    test_df = test_df[test_df["has_temporal"]].reset_index(drop=True)
else:
    print("[INFO] No full test data available; temporal validation is demonstrated only.")

print("Temporal input logic: input_1..input_3 + current target image -> mask(target)")

**Explanation:**  
This is the key temporal-data check.  
Only samples with all required temporal frames are used for ConvLSTM training and evaluation in the full pipeline.

## 6. Loaders, augmentation and generators

This section recreates the standard loading logic:

- load target image;
- load mask;
- load sequence of previous frames plus current target;
- apply consistent augmentation to image, sequence and mask during training.

In [ ]:
def load_gray_float(path: Path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr[..., None]


def load_mask_float(path: Path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.NEAREST)
    arr = (np.asarray(img, dtype=np.float32) > 127).astype(np.float32)
    return arr[..., None]


def load_temporal_sequence(row, img_size=IMG_SIZE):
    frames = []
    seq_dir = frame_dir_for_base(row["base_stem"])

    for i in range(1, HIST_T + 1):
        frames.append(load_gray_float(seq_dir / f"input_{i}.png", img_size=img_size))

    if USE_CURRENT_FRAME:
        frames.append(load_gray_float(row["image_path"], img_size=img_size))

    return np.stack(frames, axis=0)


def apply_simple_augmentation(x_img, x_seq, y_mask, do_hflip=False, do_vflip=False, rot_k=0):
    if do_hflip:
        x_img = np.flip(x_img, axis=1)
        x_seq = np.flip(x_seq, axis=2)
        y_mask = np.flip(y_mask, axis=1)

    if do_vflip:
        x_img = np.flip(x_img, axis=0)
        x_seq = np.flip(x_seq, axis=1)
        y_mask = np.flip(y_mask, axis=0)

    if rot_k:
        x_img = np.rot90(x_img, k=rot_k, axes=(0, 1))
        x_seq = np.rot90(x_seq, k=rot_k, axes=(1, 2))
        y_mask = np.rot90(y_mask, k=rot_k, axes=(0, 1))

    return x_img.copy(), x_seq.copy(), y_mask.copy()


class BaselineGenerator(Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE_BASELINE, shuffle=False, augment=False):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size)) if len(self.df) else 0

    def on_epoch_end(self):
        if self.shuffle and len(self.indices):
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        xs, ys = [], []

        for i in batch_idx:
            row = self.df.iloc[i]
            x = load_gray_float(row["image_path"])
            y = load_mask_float(row["mask_path"])

            if self.augment:
                do_h = random.random() < AUG_HFLIP_PROB
                do_v = random.random() < AUG_VFLIP_PROB
                rot_k = random.choice([0, 1, 2, 3]) if random.random() < AUG_EXTRA_ROT90_PROB else 0
                dummy_seq = np.stack([x] * T_STEPS, axis=0)
                x, _, y = apply_simple_augmentation(x, dummy_seq, y, do_h, do_v, rot_k)

            xs.append(x)
            ys.append(y)

        return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


class TemporalGenerator(Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE_CONVLSTM, shuffle=False, augment=False):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size)) if len(self.df) else 0

    def on_epoch_end(self):
        if self.shuffle and len(self.indices):
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        xs, ys = [], []

        for i in batch_idx:
            row = self.df.iloc[i]
            target = load_gray_float(row["image_path"])
            seq = load_temporal_sequence(row)
            y = load_mask_float(row["mask_path"])

            if self.augment:
                do_h = random.random() < AUG_HFLIP_PROB
                do_v = random.random() < AUG_VFLIP_PROB
                rot_k = random.choice([0, 1, 2, 3]) if random.random() < AUG_EXTRA_ROT90_PROB else 0
                target, seq, y = apply_simple_augmentation(target, seq, y, do_h, do_v, rot_k)

            xs.append(seq)
            ys.append(y)

        return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


if len(train_df) and len(test_df):
    train_base_gen = BaselineGenerator(train_df, shuffle=True, augment=AUGMENT_TRAIN)
    train_temp_gen = TemporalGenerator(train_df, shuffle=True, augment=AUGMENT_TRAIN)
    val_base_gen = BaselineGenerator(test_df, shuffle=False, augment=False)
    val_temp_gen = TemporalGenerator(test_df, shuffle=False, augment=False)
    print("Generators created from full data.")
else:
    train_base_gen = train_temp_gen = val_base_gen = val_temp_gen = None
    print("[INFO] Full-data generators are not created because full train/test data are not included.")

**Explanation:**  
The generator structure matches the original pipeline: baseline batches contain single images, while ConvLSTM batches contain temporal sequences.  
In this repository version the classes are present, but full-data generator instances are created only if the full dataset is supplied.

## 7. Dataset and preprocessing sanity checks

In the full experiment this section verifies folder sizes, generator order and mask consistency.  
Here it is kept as a safe diagnostic block.

In [ ]:
def count_files(folder: Path):
    return len(list_image_files(folder))

for d in [TRAIN_DIR, TEST_DIR, FRAMES_DIR, FINAL_IMAGES_DIR, FINAL_MASKS_DIR, FINAL_SEQUENCES_DIR]:
    print(f"{d}: exists={d.exists()} | image files={count_files(d)}")

if val_base_gen is not None and len(val_base_gen) > 0:
    xb_val, yb_val = val_base_gen[0]
    print("Validation batch shape:", xb_val.shape, yb_val.shape)
else:
    print("[INFO] Full validation generator not available in the demo repository.")

**Explanation:**  
This block would catch common dataset problems before training, such as missing masks, missing temporal frames or wrong generator order.

## 8. Quick visual check

The original notebook visualized one training sequence before training.  
Here the cell runs only if the full training generator is available.

In [ ]:
if train_base_gen is not None and train_temp_gen is not None and len(train_base_gen) and len(train_temp_gen):
    x_base, y = train_base_gen[0]
    x_temp, _ = train_temp_gen[0]

    fig, axes = plt.subplots(1, 6, figsize=(22, 4))
    axes[0].imshow(x_temp[0, 0, ..., 0], cmap="gray"); axes[0].set_title("input_1")
    axes[1].imshow(x_temp[0, 1, ..., 0], cmap="gray"); axes[1].set_title("input_2")
    axes[2].imshow(x_temp[0, 2, ..., 0], cmap="gray"); axes[2].set_title("input_3")
    axes[3].imshow(x_temp[0, 3, ..., 0], cmap="gray"); axes[3].set_title("target in sequence")
    axes[4].imshow(x_base[0, ..., 0], cmap="gray"); axes[4].set_title("baseline target")
    axes[5].imshow(y[0, ..., 0], cmap="gray"); axes[5].set_title("mask")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Quick visual check skipped because full training data are not included.")

**Explanation:**  
This visualization ensures that the sequence frames, target image and mask correspond to the same sample.

## 9. Callbacks and model construction

This section defines the same callbacks and builds both models with the standard parameters.

In [ ]:
BASE_OUT = OUT_DIR / "baseline_scss_net"
TEMP_OUT = OUT_DIR / "convlstm_scss_net"
BASE_OUT.mkdir(parents=True, exist_ok=True)
TEMP_OUT.mkdir(parents=True, exist_ok=True)


def make_callbacks(out_dir: Path):
    return [
        callbacks.ModelCheckpoint(
            filepath=str(out_dir / "best.weights.h5"),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=1,
        ),
        callbacks.CSVLogger(str(out_dir / "history.csv")),
    ]


def build_baseline_model():
    try:
        model = scss_net(
            input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
            filters=FILTERS,
            layers=LAYERS,
            batch_norm=BATCH_NORM,
            drop_prob=DROP_PROB,
        )
    except TypeError:
        model = scss_net(input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss=LOSS_FN,
        metrics=[dice_soft, iou_soft],
    )
    return model


def build_temporal_model():
    try:
        model = scss_net_convlstm_early(
            input_shape=(T_STEPS, IMG_SIZE, IMG_SIZE, CHANNELS),
            filters=FILTERS,
            layers=LAYERS,
            batch_norm=BATCH_NORM,
            drop_prob=DROP_PROB,
            convlstm_filters=CONVLSTM_FILTERS,
            convlstm_kernel=CONVLSTM_KERNEL,
            convlstm_dropout=CONVLSTM_DROPOUT,
            convlstm_recurrent_dropout=CONVLSTM_REC_DROPOUT,
            post_convlstm_conv=True,
        )
    except TypeError:
        model = scss_net_convlstm_early(
            input_shape=(T_STEPS, IMG_SIZE, IMG_SIZE, CHANNELS)
        )

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss=LOSS_FN,
        metrics=[dice_soft, iou_soft],
    )
    return model


tf.keras.backend.clear_session()
baseline_model_for_training = build_baseline_model()
temporal_model_for_training = build_temporal_model()

print("Baseline model built.")
print("ConvLSTM model built.")

**Explanation:**  
The models are constructed exactly where they would be constructed in the full notebook.  
The next section would normally start training.

## 10. Training stage

This is the original training point of the pipeline.  
In the repository version it is disabled by default, because the full training dataset is not included.

In [ ]:
history_base = None
history_temp = None

if RUN_FULL_TRAINING:
    if train_base_gen is None or train_temp_gen is None:
        raise RuntimeError("Full training requested, but full train/test generators are not available.")

    history_base = baseline_model_for_training.fit(
        train_base_gen,
        validation_data=val_base_gen,
        epochs=EPOCHS,
        callbacks=make_callbacks(BASE_OUT),
        verbose=1,
    )
    pd.DataFrame(history_base.history).to_csv(BASE_OUT / "history_final.csv", index=False)
    baseline_model_for_training.save(BASE_OUT / "final_model.keras")

    history_temp = temporal_model_for_training.fit(
        train_temp_gen,
        validation_data=val_temp_gen,
        epochs=EPOCHS,
        callbacks=make_callbacks(TEMP_OUT),
        verbose=1,
    )
    pd.DataFrame(history_temp.history).to_csv(TEMP_OUT / "history_final.csv", index=False)
    temporal_model_for_training.save(TEMP_OUT / "final_model.keras")
else:
    print("Training is skipped in this GitHub demonstration notebook.")
    print("In the original experiment, both models were trained here on the full dataset.")
    print("For final prediction below, saved trained models from trained_models/ are loaded instead.")

**Explanation:**  
This preserves the normal notebook flow, but replaces the expensive full training with an explicit explanation.  
The saved `.keras` files are used as the output of this training stage.

## 11. Training curves

In the original pipeline this section plotted loss, Dice and IoU curves.  
Here it runs only if full training was executed.

In [ ]:
def plot_history_pair(h1, h2, name1="Baseline", name2="ConvLSTM", out_path=None):
    if hasattr(h1, "history"):
        h1 = h1.history
    if hasattr(h2, "history"):
        h2 = h2.history

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for h, label in [(h1, name1), (h2, name2)]:
        axes[0].plot(h["loss"], label=f"{label} train")
        axes[0].plot(h["val_loss"], linestyle="--", label=f"{label} val")
        if "dice_soft" in h and "val_dice_soft" in h:
            axes[1].plot(h["dice_soft"], label=f"{label} train")
            axes[1].plot(h["val_dice_soft"], linestyle="--", label=f"{label} val")
        if "iou_soft" in h and "val_iou_soft" in h:
            axes[2].plot(h["iou_soft"], label=f"{label} train")
            axes[2].plot(h["val_iou_soft"], linestyle="--", label=f"{label} val")

    axes[0].set_title("Loss")
    axes[1].set_title("Dice soft")
    axes[2].set_title("IoU soft")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()


if history_base is not None and history_temp is not None:
    plot_history_pair(history_base, history_temp, out_path=OUT_DIR / "training_curves_comparison.png")
else:
    print("[INFO] Training curves skipped because full training was not executed.")

**Explanation:**  
The plotting function is retained for full reproducibility if the complete dataset is later added.

## 12. Original test split evaluation

In the full pipeline, both trained models are evaluated on the original test split.  
This section is kept but disabled by default.

In [ ]:
def predict_generator_binary(model, gen, threshold=THRESHOLD):
    probs = model.predict(gen, verbose=1)
    return (probs > threshold).astype(np.float32), probs


def metrics_summary(y_true, y_pred):
    dices, ious = [], []
    for yt, yp in zip(y_true, y_pred):
        yt2 = yt[..., 0].astype(np.float32)
        yp2 = yp[..., 0].astype(np.float32)
        dices.append(dice_np(yt2, yp2))
        ious.append(iou_np(yt2, yp2))
    return {
        "dice_mean": float(np.mean(dices)),
        "dice_std": float(np.std(dices)),
        "iou_mean": float(np.mean(ious)),
        "iou_std": float(np.std(ious)),
    }


if RUN_FULL_TEST_EVALUATION:
    if val_base_gen is None or val_temp_gen is None:
        raise RuntimeError("Full test evaluation requested, but validation generators are unavailable.")

    y_true_test = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        y_true_test.append(load_mask_float(row["mask_path"]))
    y_true_test = np.asarray(y_true_test, dtype=np.float32)

    base_pred_test, base_prob_test = predict_generator_binary(baseline_model_for_training, val_base_gen)
    temp_pred_test, temp_prob_test = predict_generator_binary(temporal_model_for_training, val_temp_gen)

    test_summary = pd.DataFrame([
        {"model": "baseline_scss_net", **metrics_summary(y_true_test, base_pred_test)},
        {"model": "convlstm_scss_net", **metrics_summary(y_true_test, temp_pred_test)},
    ])
    display(test_summary)
    test_summary.to_csv(OUT_DIR / "test_split_metrics_summary.csv", index=False)
else:
    print("[INFO] Original test split evaluation skipped because the full test dataset is not included.")

**Explanation:**  
This corresponds to the original test evaluation cell.  
The actual executable part of this repository starts in the next final-prediction section.

## 13. Final 2021 prediction with saved trained models and demo data

From this point onward the notebook uses repository demo data and saved trained models.

For AR, the final demonstration keeps only the corrected lower-threshold prediction variant with threshold `0.25`.

In [ ]:
# =========================
# UNPACK DEMO FINAL DATA
# =========================
def unzip_if_needed(zip_path: Path, extract_dir: Path):
    extract_dir.mkdir(parents=True, exist_ok=True)
    existing = [p for p in extract_dir.rglob("*") if p.is_file()]
    if existing:
        print(f"Already extracted: {extract_dir} ({len(existing)} files)")
        return extract_dir

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing ZIP file: {zip_path}")

    print(f"Extracting {zip_path.name} -> {extract_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    return extract_dir


DEMO_EXTRACT_DIR = OUT_DIR / "demo_final_2021_data"

DEMO_IMAGES_DIR = unzip_if_needed(DEMO_IMAGES_ZIP, DEMO_EXTRACT_DIR / "images")
DEMO_MASKS_DIR = unzip_if_needed(DEMO_MASKS_ZIP, DEMO_EXTRACT_DIR / "masks")
DEMO_SEQUENCES_DIR = unzip_if_needed(DEMO_SEQUENCES_ZIP, DEMO_EXTRACT_DIR / "sequences")

print("Demo image files:", len(list_image_files(DEMO_IMAGES_DIR)))
print("Demo mask files:", len(list_image_files(DEMO_MASKS_DIR)))
print("Demo sequence frame files:", len(list_image_files(DEMO_SEQUENCES_DIR)))

**Explanation:**  
In the full experiment this section would use the final prediction folders from the project dataset.  
Here the final prediction set is unpacked from compact ZIP files stored in the repository.

## 14. Load saved trained models

The saved models represent the result of the training stage that was skipped above.

In [ ]:
BASELINE_MODEL_PATH = TRAINED_MODELS_DIR / "AR_base_model.keras"
CONVLSTM_MODEL_PATH = TRAINED_MODELS_DIR / "AR_ConvLstm_model.keras"

if not BASELINE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model: {BASELINE_MODEL_PATH}")
if not CONVLSTM_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model: {CONVLSTM_MODEL_PATH}")

custom_objects = {
    "dice_soft": dice_soft,
    "iou_soft": iou_soft,
    "bce_dice_loss": bce_dice_loss,
}

baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH, custom_objects=custom_objects)
temporal_model = tf.keras.models.load_model(CONVLSTM_MODEL_PATH, custom_objects=custom_objects)

print("Loaded baseline model:", BASELINE_MODEL_PATH)
print("Loaded ConvLSTM model:", CONVLSTM_MODEL_PATH)

**Explanation:**  
The loaded models are used exactly like freshly trained models would be used in the original notebook.

## 15. Find final demo samples

This block matches target images, masks and temporal sequences.  
It supports both folder-per-sequence and flat sequence ZIP structures.

In [ ]:
def normalize_key(path: Path):
    stem = path.stem
    stem = stem.replace("_mask", "").replace("_image", "").replace("_target", "")
    stem = stem.replace("baseline_", "").replace("convlstm_", "")
    return stem


def find_sequence_folders(root: Path):
    return sorted([
        p for p in root.rglob("*")
        if p.is_dir() and all((p / f"input_{i}.png").exists() for i in range(1, HIST_T + 1))
    ])


def load_demo_sequence(sequence_folder: Path, target_image_path: Path):
    frames = []
    for i in range(1, HIST_T + 1):
        frames.append(load_gray_float(sequence_folder / f"input_{i}.png"))
    if USE_CURRENT_FRAME:
        frames.append(load_gray_float(target_image_path))
    return np.stack(frames, axis=0)


demo_image_files = list_image_files(DEMO_IMAGES_DIR)
demo_mask_files = list_image_files(DEMO_MASKS_DIR)
sequence_folders = find_sequence_folders(DEMO_SEQUENCES_DIR)

image_map = {normalize_key(p): p for p in demo_image_files}
mask_map = {normalize_key(p): p for p in demo_mask_files}

common_keys = sorted(set(image_map.keys()) & set(mask_map.keys()))

if not common_keys:
    print("[WARN] Could not match images and masks by filename. Falling back to sorted pairing.")
    n = min(len(demo_image_files), len(demo_mask_files))
    common_keys = [f"sample_{i:04d}" for i in range(n)]
    image_map = {common_keys[i]: demo_image_files[i] for i in range(n)}
    mask_map = {common_keys[i]: demo_mask_files[i] for i in range(n)}

if sequence_folders:
    n_samples = min(len(common_keys), len(sequence_folders))
else:
    n_samples = len(common_keys)
    print("[WARN] No folder-per-sequence structure detected. Target image will be repeated as sequence fallback.")

common_keys = common_keys[:n_samples]

final_rows = []
for i, key in enumerate(common_keys):
    final_rows.append({
        "sample_id": key,
        "image_path": image_map[key],
        "mask_path": mask_map[key],
        "sequence_dir": sequence_folders[i] if sequence_folders else None,
    })

final_df = pd.DataFrame(final_rows)
print("Final demo samples:", len(final_df))
display(final_df.head())

**Explanation:**  
The final prediction stage needs aligned target images, masks and sequences.  
This is the same role as the `find_final_samples()` logic in the original notebooks.

## 16. Final prediction and post-processing

The final prediction stage produces:

- probability maps;
- binary masks;
- overlays;
- metrics table.

In [ ]:
X_base_final = []
X_temp_final = []
Y_final = []
sample_ids = []

for _, row in final_df.iterrows():
    target = load_gray_float(row["image_path"])
    mask = load_mask_float(row["mask_path"])

    if row["sequence_dir"] is not None:
        seq = load_demo_sequence(row["sequence_dir"], row["image_path"])
    else:
        seq = np.stack([target] * T_STEPS, axis=0)

    X_base_final.append(target)
    X_temp_final.append(seq)
    Y_final.append(mask)
    sample_ids.append(row["sample_id"])

X_base_final = np.asarray(X_base_final, dtype=np.float32)
X_temp_final = np.asarray(X_temp_final, dtype=np.float32)
Y_final = np.asarray(Y_final, dtype=np.float32)

print("X_base_final:", X_base_final.shape)
print("X_temp_final:", X_temp_final.shape)
print("Y_final:", Y_final.shape)

base_prob_final = baseline_model.predict(X_base_final, verbose=1)
temp_prob_final = temporal_model.predict(X_temp_final, verbose=1)

base_pred_final = (base_prob_final > THRESHOLD).astype(np.float32)
temp_pred_final = (temp_prob_final > THRESHOLD).astype(np.float32)

print("Final threshold:", THRESHOLD)

**Explanation:**  
This is the first fully executable prediction stage in the GitHub demo.  
It uses saved trained models and compact demo final data.

## 17. Final metrics and diagnostics

Metrics are computed against the available masks.  
Additionally, model-to-model comparison is calculated because visual comparison between baseline and ConvLSTM predictions was important in the thesis.

In [ ]:
def model_to_model_iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float((inter + 1e-7) / (union + 1e-7))


rows = []
for i, sample_id in enumerate(sample_ids):
    y = Y_final[i, ..., 0]
    bp = base_pred_final[i, ..., 0]
    tp = temp_pred_final[i, ..., 0]

    rows.append({
        "sample_id": sample_id,
        "baseline_dice": dice_np(y, bp),
        "baseline_iou": iou_np(y, bp),
        "convlstm_dice": dice_np(y, tp),
        "convlstm_iou": iou_np(y, tp),
        "baseline_area_ratio": float(bp.mean()),
        "convlstm_area_ratio": float(tp.mean()),
        "delta_area_convlstm_minus_baseline": float(tp.mean() - bp.mean()),
        "iou_between_models": model_to_model_iou(bp, tp),
        "xor_ratio": float(np.logical_xor(bp.astype(bool), tp.astype(bool)).mean()),
    })

final_metrics_df = pd.DataFrame(rows)
metrics_path = OUT_DIR / "final_2021_demo_metrics.csv"
final_metrics_df.to_csv(metrics_path, index=False)

display(final_metrics_df.head())
display(final_metrics_df.describe())
print("Saved:", metrics_path)

**Explanation:**  
This table is a compact version of the analysis used in the thesis: standard metrics, predicted area and direct baseline-vs-ConvLSTM comparison.

## 18. Visual final comparison and collages

The visualization follows the same idea as the thesis comparison collages.

Color coding:

- red = baseline only;
- cyan = ConvLSTM only;
- yellow = both models.

In [ ]:
def difference_rgb(base_mask, conv_mask):
    base = np.squeeze(base_mask).astype(bool)
    conv = np.squeeze(conv_mask).astype(bool)

    rgb = np.zeros((base.shape[0], base.shape[1], 3), dtype=np.uint8)
    rgb[base & ~conv] = [255, 0, 0]
    rgb[conv & ~base] = [0, 180, 255]
    rgb[base & conv] = [255, 255, 0]
    return rgb


def plot_final_comparison(i):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[0].set_title("Image")

    axes[1].imshow(Y_final[i, ..., 0], cmap="gray")
    axes[1].set_title("True mask")

    axes[2].imshow(base_pred_final[i, ..., 0], cmap="gray")
    axes[2].set_title("Baseline pred")

    axes[3].imshow(temp_pred_final[i, ..., 0], cmap="gray")
    axes[3].set_title("ConvLSTM pred")

    axes[4].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[4].imshow(difference_rgb(base_pred_final[i], temp_pred_final[i]), alpha=0.55)
    axes[4].set_title("Overlay comparison")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(sample_ids[i])
    plt.tight_layout()
    plt.show()


for i in range(min(5, len(sample_ids))):
    plot_final_comparison(i)

**Explanation:**  
This reproduces the visual inspection workflow: original image, true mask, both model predictions and a direct overlay comparison.

## 19. Save final prediction outputs

This section saves binary masks and visual comparisons to the output folder.

In [ ]:
VIS_DIR = OUT_DIR / "final_visual_comparisons"
BASE_MASK_DIR = OUT_DIR / "baseline_masks"
TEMP_MASK_DIR = OUT_DIR / "convlstm_masks"

VIS_DIR.mkdir(parents=True, exist_ok=True)
BASE_MASK_DIR.mkdir(parents=True, exist_ok=True)
TEMP_MASK_DIR.mkdir(parents=True, exist_ok=True)

for i, sample_id in enumerate(sample_ids):
    base_img = Image.fromarray((base_pred_final[i, ..., 0] * 255).astype(np.uint8))
    temp_img = Image.fromarray((temp_pred_final[i, ..., 0] * 255).astype(np.uint8))

    base_img.save(BASE_MASK_DIR / f"{sample_id}_baseline_pred.png")
    temp_img.save(TEMP_MASK_DIR / f"{sample_id}_convlstm_pred.png")

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(X_base_final[i, ..., 0], cmap="gray"); axes[0].set_title("Image")
    axes[1].imshow(Y_final[i, ..., 0], cmap="gray"); axes[1].set_title("True mask")
    axes[2].imshow(base_pred_final[i, ..., 0], cmap="gray"); axes[2].set_title("Baseline pred")
    axes[3].imshow(temp_pred_final[i, ..., 0], cmap="gray"); axes[3].set_title("ConvLSTM pred")
    axes[4].imshow(X_base_final[i, ..., 0], cmap="gray")
    axes[4].imshow(difference_rgb(base_pred_final[i], temp_pred_final[i]), alpha=0.55)
    axes[4].set_title("Overlay comparison")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(sample_id)
    plt.tight_layout()
    plt.savefig(VIS_DIR / f"{sample_id}_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

print("Saved baseline masks:", BASE_MASK_DIR)
print("Saved ConvLSTM masks:", TEMP_MASK_DIR)
print("Saved visual comparisons:", VIS_DIR)

**Explanation:**  
The final prediction outputs are saved in the same spirit as the original experiment folders: separate prediction masks and visual comparison images.

## 20. Save run configuration

The run configuration is stored as JSON for reproducibility.

In [ ]:
config = {
    "phenomenon": PHENOMENON,
    "channel": CHANNEL,
    "source_folder": SOURCE_FOLDER,
    "train_dir": str(TRAIN_DIR),
    "test_dir": str(TEST_DIR),
    "frames_dir": str(FRAMES_DIR),
    "demo_data_dir": str(DEMO_DATA_DIR),
    "out_dir": str(OUT_DIR),
    "img_size": IMG_SIZE,
    "hist_t": HIST_T,
    "use_current_frame": USE_CURRENT_FRAME,
    "t_steps": T_STEPS,
    "filters": FILTERS,
    "layers": LAYERS,
    "batch_norm": BATCH_NORM,
    "drop_prob": DROP_PROB,
    "convlstm_filters": CONVLSTM_FILTERS,
    "batch_size_baseline": BATCH_SIZE_BASELINE,
    "batch_size_convlstm": BATCH_SIZE_CONVLSTM,
    "epochs": EPOCHS,
    "lr": LR,
    "threshold": THRESHOLD,
    "loss": LOSS_FN.__name__,
    "run_full_training": RUN_FULL_TRAINING,
    "run_full_test_evaluation": RUN_FULL_TEST_EVALUATION,
    "timestamp": datetime.now().isoformat(),
}

with open(OUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("All outputs saved to:", OUT_DIR)

**Explanation:**  
This mirrors the final configuration-saving cell from the original notebooks.